# Terra Signal: tratamento de valores vazios (Grupo 1)

Estudo de quais variáveis têm valores vazios e de como tratá-los antes da discretização (notebook 07) e da segmentação (notebook 08).

Cada pergunta tem três células: **pergunta** → **código** → **resposta**.

- Dados: `history.csv` e `inference.csv`, na raiz do repositório. As duas bases precisam seguir as mesmas regras.
- `CustomerFeedback` fica de fora como preditora e serve apenas para conferir `tenure`.
- Roda com pandas, numpy, scipy, statsmodels e scikit-learn, sem Spark. Funciona localmente e no Databricks (Git folder).

## Setup

In [1]:
%pip install -q statsmodels  # o ambiente serverless do Databricks pode não ter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [2]:
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

DATA_DIR = "../data/raw/"  # este notebook fica em notebooks/; os CSVs ficam em data/raw/
hist_raw = pd.read_csv(DATA_DIR + "history.csv")
inf_raw = pd.read_csv(DATA_DIR + "inference.csv")
y_raw = (hist_raw["Churn"] == "Yes").astype(int)
print("history:", hist_raw.shape, "| inference:", inf_raw.shape)

history: (5634, 23) | inference: (1409, 22)


### 1. Quais colunas têm valores vazios?

In [3]:
DISFARCADOS = ["", "unknown", "n/a", "na", "none", "null", "?", "-"]


def contar(d):
    return pd.DataFrame({
        "nulos": d.isna().sum(),
        "disfarcados": d.apply(lambda s: s.dropna().astype(str).str.strip().str.lower().isin(DISFARCADOS).sum()),
    })


tab = contar(hist_raw).join(contar(inf_raw), lsuffix="_history", rsuffix="_inference", how="outer").fillna(0).astype(int)
print(tab[tab.sum(axis=1) > 0].to_string())
print("\nColunas sem nenhum vazio:", (tab.sum(axis=1) == 0).sum(), "de", len(tab))

              nulos_history  disfarcados_history  nulos_inference  disfarcados_inference
TotalCharges            278                    0               64                      0
tenure                    0                   19                0                      4

Colunas sem nenhum vazio: 21 de 23


**Resposta:** só duas das 23 colunas têm vazios, e nas duas bases: `TotalCharges` tem nulos (278 no history e 64 no inference), e `tenure` tem o texto `unknown` (19 e 4).

### 2. Como os vazios se sobrepõem?

In [4]:
linhas = {}
for nome, d in [("history", hist_raw), ("inference", inf_raw)]:
    t = pd.to_numeric(d["tenure"], errors="coerce")
    desconhecido = d["tenure"].astype(str) == "unknown"
    nulo = d["TotalCharges"].isna()
    linhas[nome] = {
        "tenure = unknown": desconhecido.sum(),
        "TotalCharges nulo": nulo.sum(),
        "os dois vazios": (desconhecido & nulo).sum(),
        "tenure = 0": (t == 0).sum(),
        "tenure = 0 e TotalCharges nulo": ((t == 0) & nulo).sum(),
        "TotalCharges nulo com tenure > 0": ((t > 0) & nulo).sum(),
    }
print(pd.DataFrame(linhas).to_string())
print()
print(hist_raw.loc[(hist_raw["tenure"] == "unknown") & hist_raw["TotalCharges"].isna(),
                   ["customerID", "MonthlyCharges", "Contract", "Churn"]].to_string())

                                  history  inference
tenure = unknown                       19          4
TotalCharges nulo                     278         64
os dois vazios                          1          0
tenure = 0                             10          1
tenure = 0 e TotalCharges nulo         10          1
TotalCharges nulo com tenure > 0      267         63

      customerID  MonthlyCharges  Contract Churn
1097  9651-GTSAQ           106.1  One year   Yes


**Resposta:** os dois vazios quase nunca coincidem, o que permite recuperar um a partir do outro. Há três situações diferentes:

- **`TotalCharges` nulo com `tenure = 0`:** 10 no history e 1 no inference. É estrutural, porque o cliente ainda não foi cobrado.
- **`TotalCharges` nulo com `tenure > 0`:** 267 e 63. É a perda de dados relatada pela TS.
- **Os dois vazios ao mesmo tempo:** 1 linha no history (`9651-GTSAQ`) e nenhuma no inference. Ela não tem como ser recuperada pelas colunas.

### 3. Os nulos de `TotalCharges` se concentram em algum perfil?

In [5]:
t_raw = pd.to_numeric(hist_raw["tenure"], errors="coerce")
perda = hist_raw["TotalCharges"].isna() & (t_raw != 0)  # o tenure = 0 é estrutural: ainda não houve cobrança
base = t_raw != 0
for c in ["Contract", "InternetService", "PaymentMethod", "SeniorCitizen", "gender"]:
    taxas = perda[base].groupby(hist_raw.loc[base, c]).mean().round(3).to_dict()
    p = chi2_contingency(pd.crosstab(hist_raw.loc[base, c], perda[base]))[1]
    print(f"{c:16s} p = {p:.3f} | {taxas}")

Contract         p = 0.149 | {'Month-to-month': 0.053, 'One year': 0.043, 'Two year': 0.04}
InternetService  p = 0.057 | {'DSL': 0.041, 'Fiber optic': 0.055, 'No': 0.042}
PaymentMethod    p = 0.106 | {'Bank transfer (automatic)': 0.057, 'Credit card (automatic)': 0.037, 'Electronic check': 0.051, 'Mailed check': 0.044}
SeniorCitizen    p = 0.580 | {0: 0.047, 1: 0.052}
gender           p = 0.589 | {'Female': 0.049, 'Male': 0.046}


**Resposta:** não há concentração clara. As taxas de nulos ficam entre 3,7% e 5,7% em todos os grupos, e nenhuma das 5 variáveis atinge p < 0,05 (a mais próxima é `InternetService`, com p = 0,057). Isso é coerente com a perda de dados relatada pela TS, embora não prove que ela seja aleatória.

### 4. Ter `TotalCharges` nulo está ligado ao churn?

In [6]:
print("Churn com nulo:", round(y_raw[perda].mean(), 3), f"(n = {perda.sum()})", "| sem nulo:", round(y_raw[base & ~perda].mean(), 3))
print("Qui-quadrado sem controle: p =", round(chi2_contingency(pd.crosstab(perda[base], y_raw[base]))[1], 4))

t_rec = t_raw.fillna((hist_raw["TotalCharges"] / hist_raw["MonthlyCharges"]).round())
print("\nPor faixa de tenure:")
for lo, hi in [(1, 6), (7, 24), (25, 72)]:
    m = t_rec.between(lo, hi)
    print(f"  {lo:2d}-{hi:2d} meses: nulo {y_raw[m & perda].mean():.3f} (n = {(m & perda).sum()}) | sem nulo {y_raw[m & ~perda].mean():.3f}")

# Dummies provisórias: a versão final e sua justificativa ficam no notebook 07.

DUMMIES_CATEG = {
    "gender": ["Female"],
    "SeniorCitizen": [1],
    "Partner": ["Yes"],
    "Dependents": ["Yes"],
    "MultipleLines": ["Yes", "No phone service"],
    "InternetService": ["DSL", "No"],
    "OnlineSecurity": ["Yes"],
    "OnlineBackup": ["Yes"],
    "DeviceProtection": ["Yes"],
    "TechSupport": ["Yes"],
    "StreamingTV": ["Yes"],
    "StreamingMovies": ["Yes"],
    "Contract": ["One year", "Two year"],
    "PaperlessBilling": ["No"],
    "PaymentMethod": ["Mailed check", "Credit card (automatic)", "Bank transfer (automatic)"],
}


def dummies_categoricas(d):
    """Uma coluna 0/1 para cada categoria listada. A categoria omitida é a referência."""
    return pd.DataFrame(
        {f"{col}={val}": (d[col] == val).astype(int) for col, vals in DUMMIES_CATEG.items() for val in vals},
        index=d.index,
    )


ok = t_rec.notna()
X = dummies_categoricas(hist_raw[ok]).astype(float)
X = X.join(pd.get_dummies(pd.qcut(t_rec[ok], 10, duplicates="drop"), prefix="t", drop_first=True).astype(float))
X.columns = X.columns.astype(str)
X["TotalCharges_nulo"] = perda[ok].astype(float)
ajuste = sm.Logit(y_raw[ok], sm.add_constant(X)).fit(disp=0)
ic = np.exp(ajuste.conf_int().loc["TotalCharges_nulo"])
print(f"\nControlando as demais variáveis: OR = {np.exp(ajuste.params['TotalCharges_nulo']):.2f}, IC 95% [{ic[0]:.2f}; {ic[1]:.2f}], p = {ajuste.pvalues['TotalCharges_nulo']:.3f}")

Churn com nulo: 0.343 (n = 268) | sem nulo: 0.264


Qui-quadrado sem controle: p = 0.005

Por faixa de tenure:
   1- 6 meses: nulo 0.612 (n = 67) | sem nulo 0.523
   7-24 meses: nulo 0.379 (n = 58) | sem nulo 0.311
  25-72 meses: nulo 0.197 (n = 142) | sem nulo 0.145



Controlando as demais variáveis: OR = 1.29, IC 95% [0.94; 1.77], p = 0.111


**Resposta:** há um sinal fraco, que não se sustenta depois de controlar as demais variáveis.

- **Sem controle:** o churn é de 34,3% com nulo e 26,4% sem nulo (p = 0,005).
- **Por faixa de `tenure`:** o nulo fica sempre um pouco acima, de 5 a 9 pontos percentuais.
- **Com controle:** OR = 1,29, com IC 95% de 0,94 a 1,77 (p = 0,111), então não dá para descartar a ausência de efeito.

A pergunta 7 mostra se esse sinal ajuda a prever.

### 5. `TotalCharges / MonthlyCharges` recupera bem o `tenure`?

In [7]:
for nome, d in [("history", hist_raw), ("inference", inf_raw)]:
    t = pd.to_numeric(d["tenure"], errors="coerce")
    ok = t.notna() & d["TotalCharges"].notna() & (t > 0)
    erro = ((d["TotalCharges"] / d["MonthlyCharges"]).round() - t)[ok].abs()
    print(f"{nome:9s} n = {ok.sum()} | exato {(erro == 0).mean():.1%} | até 1 mês {(erro <= 1).mean():.1%} | até 2 meses {(erro <= 2).mean():.1%} | máximo {erro.max():.0f}")

print("\nOs casos reais de unknown, conferidos com o texto:")
u = hist_raw[hist_raw["tenure"] == "unknown"]
conf = pd.DataFrame({
    "estimado": (u["TotalCharges"] / u["MonthlyCharges"]).round(),
    "meses_no_texto": u["CustomerFeedback"].str.extract(r"(\d+)\s*months?", flags=2)[0],
    "anos_no_texto": u["CustomerFeedback"].str.extract(r"([\w-]+)\s+years?", flags=2)[0],
})
print(conf.to_string())
print("\nMaior tenure conhecido:", t_raw.max())

history   n = 5338 | exato 49.9% | até 1 mês 85.6% | até 2 meses 95.5% | máximo 8
inference n = 1341 | exato 49.3% | até 1 mês 84.6% | até 2 meses 95.5% | máximo 7

Os casos reais de unknown, conferidos com o texto:
      estimado meses_no_texto anos_no_texto
1059      73.0            NaN             6
1097       NaN            NaN          four
1791      12.0             12           NaN
1984      73.0            NaN             6
2361      22.0            NaN           two
2372       1.0            NaN           NaN
2641      36.0             36           NaN
2896      67.0            NaN             6
2936      14.0             15           NaN
2959      68.0            NaN          five
3153      65.0            NaN             5
3300       7.0              6           one
3388      67.0            NaN          five
3476       5.0              4           NaN
3515       5.0              6           NaN
3837      74.0            NaN             5
4875      28.0             28       

**Resposta:** recupera bem o suficiente para discretizar, mas não com exatidão. A estimativa acerta o mês exato em cerca de 50% dos casos, erra até 1 mês em 86% e até 2 meses em 96%.

Nos casos reais de `unknown`, a recuperação funciona em 18 das 19 linhas do history e nas 4 do inference. Nos 8 casos em que o texto cita os meses, a diferença é de no máximo 1 mês. Três estimativas passam do maior `tenure` conhecido (73, 73 e 74, contra 72), mas isso não importa depois de discretizar, porque elas caem na faixa mais alta. O notebook 07 confere quantos casos caem na faixa certa.

### 6. `MonthlyCharges × tenure` estima bem o `TotalCharges`?

In [8]:
for nome, d in [("history", hist_raw), ("inference", inf_raw)]:
    t = pd.to_numeric(d["tenure"], errors="coerce")
    ok = t.notna() & d["TotalCharges"].notna() & (t > 0)
    real = d.loc[ok, "TotalCharges"]
    erro = ((d.loc[ok, "MonthlyCharges"] * t[ok] - real).abs() / real)
    erro_med = ((real.median() - real).abs() / real)
    print(f"{nome:9s} produto: mediana {erro.median():.1%}, média {erro.mean():.1%}, p95 {erro.quantile(.95):.1%} | imputar pela mediana da coluna: mediana {erro_med.median():.1%}")

ok = t_raw.notna() & hist_raw["TotalCharges"].notna() & (t_raw > 0)
erro = ((hist_raw.loc[ok, "MonthlyCharges"] * t_raw[ok] - hist_raw.loc[ok, "TotalCharges"]).abs() / hist_raw.loc[ok, "TotalCharges"])
print("\nErro mediano por faixa de tenure:")
print(erro.groupby(pd.cut(t_raw[ok], [0, 6, 24, 72])).median().round(3).to_string())

history   produto: mediana 2.0%, média 3.2%, p95 10.4% | imputar pela mediana da coluna: mediana 74.4%
inference produto: mediana 2.0%, média 3.1%, p95 10.2% | imputar pela mediana da coluna: mediana 74.0%

Erro mediano por faixa de tenure:
tenure
(0, 6]      0.017
(6, 24]     0.034
(24, 72]    0.016


**Resposta:** sim. O erro mediano é de 2,0% (3,2% em média e 10,4% no percentil 95), nas duas bases. Imputar pela mediana da coluna teria erro mediano de 74%, então essa alternativa fica descartada.

### 7. Imputar ou criar uma faixa `Ausente`?

Comparamos três opções com a mesma regressão logística e validação cruzada (5 folds, 4 repetições). `tenure` e `TotalCharges` ficam em decis calculados no treino de cada fold, de forma provisória, pois a discretização final é do notebook 07.

In [9]:
d = hist_raw[t_rec.notna()].reset_index(drop=True)
y_cv = (d["Churn"] == "Yes").astype(int)
t_cru = pd.to_numeric(d["tenure"], errors="coerce")
tc_cru = d["TotalCharges"].where(t_cru != 0, 0.0)
t_imp = t_cru.fillna((d["TotalCharges"] / d["MonthlyCharges"]).round())
tc_imp = d["TotalCharges"].fillna(d["MonthlyCharges"] * t_imp).where(t_imp != 0, 0.0)
X_cat = dummies_categoricas(d)


def decis(serie, tr, te, nome):
    """Decis calculados no treino. Se houver vazio, ele vira a faixa Ausente."""
    cortes = np.unique(np.nanquantile(serie.iloc[tr], np.arange(.1, 1, .1)))
    def monta(ix):
        s = serie.iloc[ix]
        f = pd.Series(np.digitize(s, cortes, right=True), index=s.index)
        saida = pd.DataFrame({f"{nome}_{i}": ((f == i) & s.notna()).astype(int) for i in range(1, len(cortes) + 1)})
        if serie.isna().any():
            saida[f"{nome}_ausente"] = s.isna().astype(int)
        return saida
    return monta(tr), monta(te)


def avaliar(t_serie, tc_serie):
    aucs = []
    for tr, te in RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0).split(d, y_cv):
        ta, tb = decis(t_serie, tr, te, "t")
        ca, cb = decis(tc_serie, tr, te, "tc")
        Xtr = pd.concat([X_cat.iloc[tr], ta, ca], axis=1)
        Xte = pd.concat([X_cat.iloc[te], tb, cb], axis=1)
        m = LogisticRegression(max_iter=3000).fit(Xtr, y_cv.iloc[tr])
        aucs.append(roc_auc_score(y_cv.iloc[te], m.predict_proba(Xte)[:, 1]))
    return np.array(aucs)


ref = avaliar(t_imp, tc_imp)
print(f"Imputar os dois:                 AUC {ref.mean():.4f}")
for nome, a in {"Ausente em TotalCharges": avaliar(t_imp, tc_cru), "Ausente nos dois": avaliar(t_cru, tc_cru)}.items():
    print(f"{nome:32s} AUC {a.mean():.4f} | diferença {np.mean(a - ref):+.4f} ± {np.std(a - ref):.4f}")

Imputar os dois:                 AUC 0.8462


Ausente em TotalCharges          AUC 0.8452 | diferença -0.0010 ± 0.0012
Ausente nos dois                 AUC 0.8455 | diferença -0.0007 ± 0.0016


**Resposta:** imputar é a melhor opção, embora a diferença seja pequena. A AUC é de 0,8462 com imputação, e criar a faixa `Ausente` não melhora: −0,0010 ± 0,0012 só em `TotalCharges` e −0,0007 ± 0,0016 nos dois. Como a imputação tem erro baixo e não acrescenta dummies, ela é a escolha mais simples.

### 8. As regras finais deixam alguma base com vazios?

In [10]:
def prep(d):
    """Regras do notebook 06: padroniza PhoneService, recupera tenure e imputa TotalCharges."""
    d = d.copy()
    d["PhoneService"] = d["PhoneService"].str.capitalize()
    d["tenure_num"] = pd.to_numeric(d["tenure"], errors="coerce")
    d["tenure_num"] = d["tenure_num"].fillna((d["TotalCharges"] / d["MonthlyCharges"]).round())
    d["TotalCharges_imp"] = d["TotalCharges"].fillna(d["MonthlyCharges"] * d["tenure_num"])
    d.loc[d["tenure_num"] == 0, "TotalCharges_imp"] = 0
    return d


hist = prep(hist_raw)
hist = hist[hist["tenure_num"].notna()].reset_index(drop=True)  # 1 linha sem tenure e sem TotalCharges
inf = prep(inf_raw)
y = (hist["Churn"] == "Yes").astype(int)
print("Clientes no treino:", len(hist), "| churners:", y.sum(), f"({y.mean():.1%})")


for nome, b in [("history", hist), ("inference", inf)]:
    print(nome, "| linhas:", len(b), "| vazios restantes:", b[["tenure_num", "TotalCharges_imp"]].isna().sum().to_dict())

Clientes no treino: 5633 | churners: 1503 (26.7%)
history | linhas: 5633 | vazios restantes: {'tenure_num': 0, 'TotalCharges_imp': 0}
inference | linhas: 1409 | vazios restantes: {'tenure_num': 0, 'TotalCharges_imp': 0}


**Resposta:** não. Depois das regras, nenhuma das bases tem vazios, e o treino fica com 5.633 clientes (1.503 churners).

| Situação | History | Inference | Decisão | Fundamento |
|---|---|---|---|---|
| `tenure = unknown` | 18 | 4 | `TotalCharges / MonthlyCharges`, arredondado | Erro de até 2 meses em 96% dos casos (pergunta 5) |
| `TotalCharges` nulo com `tenure = 0` | 10 | 1 | Preencher com 0 | Ainda não houve cobrança (pergunta 2) |
| `TotalCharges` nulo com `tenure > 0` | 267 | 63 | `MonthlyCharges × tenure` | Erro mediano de 2,0% (pergunta 6) |
| Os dois vazios | 1 | 0 | Retirar do treino | Não há como recuperar pelas colunas, e não ocorre no inference (pergunta 2) |

`MonthlyCharges` é usada nessas contas, mas não entra como preditora do modelo.

## Limites desta análise

- **A recuperação de `tenure` é aproximada:** reajustes e promoções afastam `TotalCharges` de `MonthlyCharges × tenure`, então a estimativa acerta o mês exato só em metade dos casos.
- **A validação usa os casos preenchidos:** ela supõe que os casos vazios se comportam como os preenchidos, o que os dados não permitem confirmar.
- **O sinal fraco do nulo pode ser real:** o intervalo de confiança da pergunta 4 ainda admite um efeito moderado. Ele só não melhorou a previsão.
- **Uso em produção:** um cliente com `tenure` e `TotalCharges` vazios ao mesmo tempo não recebe tratamento por estas regras. Se o app permitir esse caso, será preciso definir uma regra para ele.